# 🛰️ RunPod 환경 점검 — Colab `01_colab_setup` 의 RunPod 버전

Colab에서는 *런타임 → T4 GPU* 만 고르면 끝났지만, RunPod에서는 **Pod 사양 · 디스크 · 메모리 한도 · API 키 관리**를 직접 챙겨야 합니다.
이 노트북은 새 Pod에 접속하자마자 실행하는 점검표입니다.

| # | 점검 | Colab 에서는 | RunPod 에서는 |
|---|------|-------------|---------------|
| 1 | 파이썬 / 가상환경 | 고정 런타임 | 내가 고른 커널(`.venv`)인지 확인 |
| 2 | GPU | 런타임 유형 T4 | Pod 생성 시 고른 GPU |
| 3 | 컨테이너 메모리 한도 | 신경 쓸 일 없음 | **`free -h` 와 다름 — cgroup 으로 확인** |
| 4 | 디스크 | `/content` 임시 | `/`(Container) 와 `/workspace`(Volume) 구분 |
| 5 | 모델 캐시 위치 | 세션마다 재다운로드 | `HF_HOME` 을 Volume 으로 |
| 6 | API 키 | Colab Secrets (`userdata`) | 환경변수 / `.env` |

> 💡 터미널에서 한 번에 보고 싶으면 `bash scripts/check_pod.sh`

---
## 1. 파이썬 · 가상환경

In [ ]:
import sys, platform, socket

print(f"Python     : {platform.python_version()}")
print(f"실행 파일   : {sys.executable}")
print(f"호스트명    : {socket.gethostname()}  ← 터미널 프롬프트의 root@xxxx 와 같은지")
in_venv = sys.prefix != sys.base_prefix
print(f"가상환경    : {'✅ 사용 중' if in_venv else '⚠️ 시스템 파이썬 (setup_pod.sh 로 만든 커널을 선택하세요)'}")

---
## 2. GPU

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1024**3:.1f} GB")
    print(f"CUDA : {torch.version.cuda} | torch {torch.__version__}")
else:
    print("❌ GPU 없음 — CPU Pod 이거나 드라이버 문제입니다. Pod 사양을 확인하세요.")

In [ ]:
!nvidia-smi

---
## 3. 컨테이너 메모리 한도 ⚠️ 가장 중요

`free -h` 는 **서버(호스트) 전체 메모리**를 보여줍니다. 컨테이너에 실제로 허용된 양은 cgroup 에 있습니다.

> 실제 사례: `free -h` 는 251Gi 였지만 cgroup 한도는 **512MB** → VS Code 확장 호스트가 20~30초마다 SIGKILL 되며 `.ipynb` 가 열리지 않았습니다.

In [ ]:
from pathlib import Path

def read_cgroup_memory():
    v2 = Path("/sys/fs/cgroup/memory.max")
    if v2.exists():
        limit = v2.read_text().strip()
        usage = Path("/sys/fs/cgroup/memory.current").read_text().strip()
    else:
        base = Path("/sys/fs/cgroup/memory")
        limit = (base / "memory.limit_in_bytes").read_text().strip()
        usage = (base / "memory.usage_in_bytes").read_text().strip()
    limit = None if limit == "max" or int(limit) > 1 << 60 else int(limit)
    return limit, int(usage)

limit, usage = read_cgroup_memory()
gb = 1024**3
if limit is None:
    print(f"한도: 무제한  | 사용: {usage/gb:.2f} GB")
else:
    print(f"한도: {limit/gb:.2f} GB | 사용: {usage/gb:.2f} GB ({usage/limit*100:.0f}%)")
    if limit < 4 * gb:
        print("⚠️ 4GB 미만 — VS Code + Jupyter + 모델 로드를 버티기 어렵습니다. 더 큰 사양으로 Pod 를 다시 만드세요.")
    else:
        print("✅ 충분합니다.")

---
## 4. 디스크 — Container Disk vs Volume

In [ ]:
import shutil

for path, desc in [("/", "Container Disk (재시작 시 초기화)"), ("/workspace", "Volume (유지됨)")]:
    try:
        total, used, free = shutil.disk_usage(path)
        print(f"{path:<11} {desc:<28} 전체 {total/gb:6.1f} GB | 여유 {free/gb:6.1f} GB")
    except FileNotFoundError:
        print(f"{path:<11} 없음")

print("\n💡 작업 파일 · 모델 캐시 · 결과물은 항상 /workspace 아래에 두세요.")

---
## 5. 모델 캐시 위치 (`HF_HOME`)

EXAONE-3.5-2.4B 는 약 5GB 입니다. 기본 위치(`~/.cache`)는 Container Disk 라 금방 차고, Pod 를 재시작하면 다시 받아야 합니다.

In [ ]:
import os

hf_home = os.environ.get("HF_HOME")
if hf_home and hf_home.startswith("/workspace"):
    print(f"✅ HF_HOME = {hf_home}")
else:
    print(f"⚠️ HF_HOME = {hf_home or '(미설정 → ~/.cache/huggingface)'}")
    print("   터미널에서:  echo 'export HF_HOME=/workspace/.cache/huggingface' >> ~/.bashrc && source ~/.bashrc")
    print("   (이미 띄운 커널에는 반영되지 않으니 커널 재시작 필요)")

---
## 6. API 키 — Colab Secrets 대신 `.env`

```python
# Colab
from google.colab import userdata
userdata.get("OPENAI_API_KEY")      # RunPod 에서는 TimeoutException
```

RunPod 에서는 저장소 루트의 `.env` (`.env.example` 복사) 에 키를 적고 `python-dotenv` 로 읽습니다.
`.env` 는 `.gitignore` 에 들어 있어 깃에 올라가지 않습니다.

In [ ]:
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print(f".env 위치: {env_path or '찾지 못함'}")

for name in ["OPENAI_API_KEY", "HF_TOKEN"]:
    value = os.environ.get(name, "")
    print(f"{name:<15}: {'✅ 설정됨 (' + str(len(value)) + '자)' if value else '— 미설정'}")   # 키 값은 절대 출력하지 않기

In [ ]:
# (선택) OpenAI 연결 테스트 — 키가 있을 때만 실행됩니다
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()   # OPENAI_API_KEY 환경변수를 자동으로 읽음
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "RunPod 연결 테스트입니다. 한 문장으로 인사해 주세요."}],
        max_tokens=50,
    )
    print("GPT:", resp.choices[0].message.content)
else:
    print("OPENAI_API_KEY 가 없어 건너뜁니다. EXAONE 실습은 키 없이 가능합니다.")

---
## ✅ 점검 체크리스트

| # | 항목 | 기준 |
|---|------|------|
| 1 | 커널 | `Python (runpod-llm)` 가상환경 |
| 2 | GPU | 이름 · VRAM 출력 |
| 3 | 메모리 한도 | **4GB 이상** (권장 16GB+) |
| 4 | 디스크 | `/workspace` 여유 충분 |
| 5 | HF_HOME | `/workspace/...` |
| 6 | API 키 | `.env` 에서 로드, 값은 출력하지 않음 |

모두 통과하면 `01_EXAONE_챗봇_응용.ipynb` 로 넘어가세요.